In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_2020_UGA.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2020_UGA.ID_001,0,ENSG00000000003,83,0.607222,Unknown
1,2020_UGA.ID_001,0,ENSG00000000005,0,0.000000,Unknown
2,2020_UGA.ID_001,0,ENSG00000000419,2489,68.417030,Unknown
3,2020_UGA.ID_001,0,ENSG00000000457,1221,5.885515,Unknown
4,2020_UGA.ID_001,0,ENSG00000000460,532,2.958027,Unknown


In [3]:
# Unlike 2019, raw_count has actual values in 2020; material is still all 'Unknown'
# Drop material before pivoting; keep raw_count aside — we pivot on tpm_count for consistency with 2019
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())

raw_count null fraction: 0.0
material unique values: <StringArray>
['Unknown']
Length: 1, dtype: str


In [4]:
# Drop material (entirely 'Unknown'); drop raw_count to match 2019 pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2020_UGA.ID_001,0,ENSG00000000003,0.607222
1,2020_UGA.ID_001,0,ENSG00000000005,0.000000
2,2020_UGA.ID_001,0,ENSG00000000419,68.417030
3,2020_UGA.ID_001,0,ENSG00000000457,5.885515
4,2020_UGA.ID_001,0,ENSG00000000460,2.958027


In [5]:
df['timepoint'].unique()

array([ 0, 28])

In [6]:
# Filter to only timepoints 0, 7, 28. Timepoint 3 not in the challenge set
df = df[df['timepoint'].isin([0, 7, 28])]

# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "d{timepoint}_{gene}" format
df_pivot.columns = [f'd{int(tp)}_{gene}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)

(48, 112829)


,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,...,d28_ENSG00000280441,d28_ENSG00000280443,d28_ENSG00000280444,d28_ENSG00000280445,d28_ENSG00000280446,d28_ENSG00000280449,d28_ENSG00000280450,d28_ENSG00000280451,d28_ENSG00000280453,d28_ENSG00000280454
0,2020_UGA.ID_001,0.607222,0.0,68.41703,5.885515,2.958027,445.721808,0.904403,36.064293,11.176851,...,13.932189,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_pivot.head(1)

In [7]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_2020_UGA_cleaned.csv', index=False)